# FX matrix (quotes, triangulation, cross rates)

Deep-dive reference: **`FxMatrix`** for **direct** FX quotes and **policy-aware** lookups.

## Concept

Quotes are stored as **1 base = rate quote** (e.g. **EUR/USD = 1.10** means one euro buys 1.10 dollars). **Direct** pairs are those you explicitly **`set_quote`**. **Cross rates** often require a **vehicle** currency (commonly USD): compute **EUR/GBP** from **EUR/USD** and **GBP/USD** when the matrix does not supply the cross directly.

## API walkthrough

`set_quote(base, quote, rate)` then `rate(base, quote, as_of_date, FxConversionPolicy.…)` returns **`FxRateResult`** with **`.rate`** and **`.triangulated`**.

In [1]:
from datetime import date

from finstack.core.currency import Currency
from finstack.core.market_data import FxConversionPolicy, FxMatrix

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
fx.set_quote(Currency("GBP"), Currency("USD"), 1.27)
fx.set_quote(Currency("JPY"), Currency("USD"), 0.0067)

as_of = date(2024, 1, 1)
policy = FxConversionPolicy.CASHFLOW_DATE

eur_usd = fx.rate(Currency("EUR"), Currency("USD"), as_of, policy)
print("Direct EUR/USD rate:", eur_usd.rate, "triangulated:", eur_usd.triangulated)

jpy_usd = fx.rate(Currency("JPY"), Currency("USD"), as_of, policy)
print("Direct JPY/USD (USD per 1 JPY):", jpy_usd.rate)

usd_jpy = fx.rate(Currency("USD"), Currency("JPY"), as_of, policy)
print("Inverse USD/JPY (JPY per 1 USD):", usd_jpy.rate)

Direct EUR/USD rate: 1.1 triangulated: False
Direct JPY/USD (USD per 1 JPY): 0.0067
Inverse USD/JPY (JPY per 1 USD): 149.2537313432836


## Practical example

**Multi-currency setup** and **EUR/GBP** via **USD** as vehicle when the cross is not quoted directly.

In [2]:
from datetime import date

from finstack.core.currency import Currency
from finstack.core.market_data import FxConversionPolicy, FxMatrix

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
fx.set_quote(Currency("GBP"), Currency("USD"), 1.27)
fx.set_quote(Currency("CHF"), Currency("USD"), 1.12)

as_of = date(2024, 1, 1)
pol = FxConversionPolicy.CASHFLOW_DATE

try:
    eur_gbp = fx.rate(Currency("EUR"), Currency("GBP"), as_of, pol)
    print("EUR/GBP from matrix:", eur_gbp.rate, "triangulated:", eur_gbp.triangulated)
except ValueError as err:
    print("Matrix lookup EUR->GBP failed (no direct/triangulated path):", err)
    eur_usd = fx.rate(Currency("EUR"), Currency("USD"), as_of, pol).rate
    gbp_usd = fx.rate(Currency("GBP"), Currency("USD"), as_of, pol).rate
    synthetic = eur_usd / gbp_usd
    print("Synthetic EUR/GBP via USD (EUR/USD ÷ GBP/USD):", round(synthetic, 6))

try:
    chf_eur = fx.rate(Currency("CHF"), Currency("EUR"), as_of, pol)
    print("CHF/EUR from matrix:", chf_eur.rate, "triangulated:", chf_eur.triangulated)
except ValueError:
    chf_usd = fx.rate(Currency("CHF"), Currency("USD"), as_of, pol).rate
    eur_usd = fx.rate(Currency("EUR"), Currency("USD"), as_of, pol).rate
    print("Synthetic CHF/EUR via USD:", round(chf_usd / eur_usd, 6))

Matrix lookup EUR->GBP failed (no direct/triangulated path): Requested item not found: FX:EUR->GBP
Synthetic EUR/GBP via USD (EUR/USD ÷ GBP/USD): 0.866142
Synthetic CHF/EUR via USD: 1.018182


## Takeaways

- **`set_quote`** defines **base → quote** with **1 base = rate quote**.
- **`FxRateResult`** carries **`.rate`** and whether the value was **triangulated**.
- **Crosses** may need **manual synthesis** through a **vehicle** currency if the engine does not infer that path.
- **`FxConversionPolicy`** selects **how** the as-of is interpreted for more complex instruments.